In [4]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

import torch
from torch import nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import callbacks 

from src import PATH

In [5]:
from importlib import reload
from src import _helper_net, _linear
reload(_helper_net)
reload(_linear)

<module 'src._linear' from '/home/wergillius/Project/diffuse_differentiate/src/_linear.py'>

In [6]:
from torch.utils.data import DataLoader, Dataset, TensorDataset

In [7]:
adata = sc.read_h5ad("/home/wergillius/Project/diffuse_differentiate/data/Barcodelet/integrated_mesc_group0_Nov7.h5ad")
adata

AnnData object with n_obs × n_vars = 5745 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh', 'assignment', 'starting.state', 'dataset', 'control', 'cell_type', 'condition', 'dose_val', 'batch', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'dpt_pseudotime', 'RA+Wnt+Fgf', 'numeric_iloc', 'split', 'discrete_time'
    var: 'gene_name'
    uns: 'condition_colors', 'diff', 'diffmap_evals', 'dpt_changepoints', 'dpt_groups_colors', 'dpt_grouptips', 'iroot', 'neighbors', 'umap', 'unique_token_dict'
    obsm: 'Tr_SampledX_r100', 'X_diffmap', 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'diff_connectivities', 'diff_distances', 'distances'

In [4]:
adata_pc500 = sc.tl.pca(adata,n_comps=500, copy=True)

# Auto-encoder to reconstruct

In [6]:
from sklearn.preprocessing import normalize

In [5]:
perturbations = ['RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh']
assignment_C = adata.obs[perturbations].values
gene_X = adata.X

X =  np.concatenate([gene_X, assignment_C], axis=1)

Y = gene_X

train_idx = adata.obs.split != 'test'
test_idx = adata.obs.split == 'test'

x_train = torch.from_numpy(X[train_idx]).float()
x_test = torch.from_numpy(X[test_idx]).float()

Y_train = torch.from_numpy(Y[train_idx]).float()
Y_test = torch.from_numpy(Y[test_idx]).float()

# to torch dataloader
train_ds = TensorDataset(x_train, Y_train)
train_dl = DataLoader(train_ds, batch_size=32, num_workers=8, shuffle=True)

test_ds = TensorDataset(x_test, Y_test)
test_dl = DataLoader(test_ds, batch_size=32,  num_workers=8, shuffle=False)

In [ ]:
X.shape

# model

# Training CAE model

### shallow CAE

In [18]:
CAE = _linear.CAE_model(input_dim=2007, output_dim=2000, hidden=[400],
                  lr=1e-5, weight_decay=1e-9)
Trainer = pl.Trainer(accelerator='gpu', gpus=[0],
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'CAE_model_reconX'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )
Trainer.fit(CAE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

### deeper CAE

In [8]:
### v0
# CAE = _linear.CAE_model(input_dim=2007, output_dim=2000, hidden=[1024, 512,128,512, 1024],
#                   lr=1e-5, weight_decay=1e-9)

### v1 
CAE = _linear.CAE_model(input_dim=2007, output_dim=2000, hidden=[512,512,512],
                  lr=1e-5, weight_decay=1e-9)


# set up and fit
Trainer = pl.Trainer(accelerator='gpu', gpus=[0],
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'CAE_deep_model_reconX'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

Trainer.fit(CAE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

# Training Embedding AE model

## 7 conditions embedding

In [7]:
reload(_helper_net)
reload(_linear)

<module 'src._linear' from '/home/wergillius/Project/diffuse_differentiate/src/_linear.py'>

In [6]:
# Deep Emb CAE v1
Emb_AE = _linear.Embedding_model(gene_dim=2000, condition_dim=7,
                output_dim=2000, hidden=[512,512,512,519,519],
                lr=1e-5, weight_decay=1e-5)

In [7]:
Emb_AE.embedder.weight.shape

torch.Size([7, 32])

In [8]:
# set up and fit
Trainer = pl.Trainer(accelerator='gpu', gpus=[0],
                    #  fast_dev_run=True,
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'Embedding_CAE_deep_model_reconX'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

Trainer.fit(Emb_AE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:467: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/pytorch_lightning/loops/utilities.py:94: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name           | Type       | Params
----------------------------------------------
0 | loss_fn        | MSELoss    | 0     
1 | train_r2_score | R2Score    | 0     
2 | val_r2_score   | R2Score    | 0     
3 | encoder        | MLP        | 1.5 M 
4 | embedder       | Embedding  | 224   
5 | decoder        | Sequential | 1.6 M 
----------------------------------------------
3.1 M     Trainable params
0         Non-trainable params
3.1 M     Total params
12.571    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=1000` reached.


## 33 conditions Embedding

In [5]:
# perturbations = ['RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh']
assignment_C = pd.get_dummies(adata.obs['assignment'].values)
gene_X = adata.X

X = np.concatenate([gene_X, assignment_C], axis=1)
Y = gene_X

train_idx = adata.obs.split != 'test'
test_idx = adata.obs.split == 'test'

x_train = torch.from_numpy(X[train_idx]).float()
x_test = torch.from_numpy(X[test_idx]).float()

Y_train = torch.from_numpy(Y[train_idx]).float()
Y_test = torch.from_numpy(Y[test_idx]).float()

# to torch dataloader
train_ds = TensorDataset(x_train, Y_train)
train_dl = DataLoader(train_ds, batch_size=32, num_workers=8, shuffle=True)

test_ds = TensorDataset(x_test, Y_test)
test_dl = DataLoader(test_ds, batch_size=32,  num_workers=8, shuffle=False)

In [6]:
x_split = torch.tensor_split(x_train[:3,], (2000,), dim=1)

In [7]:
x_split[0].shape

torch.Size([3, 2000])

## deep Embedding AE

In [26]:
reload(_helper_net)
reload(_linear)

<module 'src._linear' from '/home/wergillius/Project/diffuse_differentiate/src/_linear.py'>

In [27]:
# To ensuer skip connection
    # 512 -> 512 
    # 512+32 -> 544
Emb_AE = _linear.Embedding_model(gene_dim=2000, condition_dim=33,
                output_dim=2000, hidden=[512,512,544],
                lr=1e-5, weight_decay=1e-9)
# set up and fit
Trainer = pl.Trainer(accelerator='gpu', gpus=[0],
                    #  fast_dev_run=True,
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'Embedding_CAE_deep_model_reconX'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

Trainer.fit(Emb_AE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

In [ ]:
list(Emb_AE.state_dict().keys())

# update chekcpoints

for model that was defined with old version, transfer their saved weights to the new CAE and Embedding model.

So that they can be normally reloaded

In [ ]:
import yaml

# load the old checkpoints
ckpt_dir = "/home/wergillius/data/diffuse_differentiate/Barcodelet/CAE_deep_model_reconX/lightning_logs"
ckpt_path = os.path.join(ckpt_dir, 'version_5/checkpoints/epoch=920-step=149202.ckpt')
checkpoint = torch.load(ckpt_path, map_location='cpu')
weight = checkpoint['state_dict']


# inititates a new model using updated kwargs
encoder_kwargs = {'dimensions':[2007, 512, 512],
                  "output_activation":"ReLU"}

decoder_kwargs = {'dimensions':[512, 512, 2000], 
                  "use_batchnorm":False, 
                  "output_activation":None}

# CAE_resume = _linear.CAE_model(
#               encoder_kwargs = encoder_kwargs, 
#               decoder_kwargs = decoder_kwargs,
#               lr=1e-5,
#               weight_decay=1e-9)

# # check out this statedict to find orderdict names
# resume_statedict_key = list(CAE_resume.state_dict().keys())

# rename weights
from collections import OrderedDict
new_weight = OrderedDict([
    ("encoder.model.0.0.weight", weight["encoder.model.0.0.weight"]),
    ("encoder.model.0.0.bias"  , weight["encoder.model.0.0.bias"]),
    ("encoder.model.1.0.weight", weight["encoder.model.1.0.weight"]),
    ("encoder.model.1.0.bias"  , weight["encoder.model.1.0.bias"]),
    ('decoder.model.0.0.weight', weight["decoder.0.model.0.0.weight"]), 
    ('decoder.model.0.0.bias'  , weight["decoder.0.model.0.0.bias"]), 
    ('decoder.model.1.0.weight', weight["decoder.1.weight"]), 
    ('decoder.model.1.0.bias'  , weight["decoder.1.bias"])
])

hparams = dict(
    encoder_kwargs = encoder_kwargs, 
    decoder_kwargs = decoder_kwargs,
    lr=1e-5,
    weight_decay=1e-9
)

# save updated hyper-params



f = open("/home/wergillius/data/diffuse_differentiate/Barcodelet/CAE_deep_model_reconX/lightning_logs/version_5/hparams.yaml", "w")
yaml.dump(hparams, f)
f.close()


# save updated checkpoint
checkpoint['state_dict'] = new_weight
checkpoint['hyper_parameters'] = hparams
torch.save(checkpoint, ckpt_path)

### update CAE   
`CVAE_model/lightning_logs/version_2/checkpoints/epoch=23-step=3888.ckpt`

In [23]:
# load the old checkpoints

# inititates a new model using updated kwargs
encoder_kwargs = {'dimensions':[2007, 2048],
                  "output_activation":"ReLU"}

decoder_kwargs = {'dimensions':[2048, 2000], 
                  "use_batchnorm":False, 
                  "output_activation":None}

hparams = dict(encoder_kwargs=encoder_kwargs,
               decoder_kwargs=decoder_kwargs,
                lr=1e-4, weight_decay=1e-4)

VAE = _linear.CAE_model(**hparams)

In [12]:
resume_statedict_key = list(VAE.state_dict().keys())
resume_statedict_key

['encoder.model.0.0.weight',
 'encoder.model.0.0.bias',
 'decoder.model.0.0.weight',
 'decoder.model.0.0.bias']

In [14]:

ckpt_path = "/home/wergillius/data/diffuse_differentiate/Barcodelet/CVAE_model/lightning_logs/version_2/checkpoints/epoch=23-step=3888.ckpt"
checkpoint = torch.load(ckpt_path, map_location='cpu')
weight = checkpoint['state_dict']


In [19]:
weight_key = list(weight.keys())[4:]
rename_dict = dict(zip(resume_statedict_key,weight_key))

In [21]:

from collections import OrderedDict
new_state_dict = []
for k, v in rename_dict.items():
    new_state_dict.append(
        (k, weight[v])
    )

new_state_dict = OrderedDict(new_state_dict)

In [24]:
# save updated checkpoint
checkpoint['state_dict'] = new_state_dict
checkpoint['hyper_parameters'] = hparams
torch.save(checkpoint, ckpt_path)

In [25]:
VAE_resume = _linear.CAE_model.load_from_checkpoint(ckpt_path)